In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/ml_assn_04/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/ml_assn_04/test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nEmotion distribution:")
print(train_df['emotion'].value_counts())
print("\nSample row:")
print(train_df.head(1))

Train shape: (28709, 2)
Test shape: (7178, 1)

Columns: ['emotion', 'pixels']

Emotion distribution:
emotion
3    7215
6    4965
4    4830
2    4097
0    3995
5    3171
1     436
Name: count, dtype: int64

Sample row:
   emotion                                             pixels
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['emotion']
)

print("Train samples:", len(train_df))
print("Val samples:", len(val_df))

Train samples: 22967
Val samples: 5742


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ochiga (ml_assn_04) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import torch.nn as nn

class DeepCNN(nn.Module):
    def __init__(self, dropout1=0.5, dropout2=0.3):
        super(DeepCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(dropout1),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout2),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.labels = dataframe['emotion'].values
        self.pixels = dataframe['pixels'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        pixels = np.array(self.pixels[idx].split(), dtype=np.float32)
        image = pixels.reshape(48, 48)

        image = image / 255
        image = torch.tensor(image).unsqueeze(0)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\nDevice:", device)


Device: cpu


In [ ]:
val_loader_bs32 = DataLoader(FERDataset(val_df), batch_size=32, shuffle=False)
deep_model = DeepCNN(dropout1=0.6, dropout2=0.5).to(device)
deep_model.load_state_dict(torch.load(
    "/content/drive/MyDrive/ml_assn_04/best_deep-cnn-cosine-lr.pt",
    map_location=device
))
deep_model.eval()

DeepCNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1

In [ ]:
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader_bs32:
        images, labels = images.to(device), labels.to(device)
        outputs = deep_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

run = wandb.init(
    entity="ml_assn_04",
    project="ml_assn_04",
    id="001zqj9i",
    resume="must"
)

run.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=all_labels, preds=all_preds, class_names=emotion_labels
    )
})

pred_table = wandb.Table(columns=["actual", "predicted_label", "predicted_idx"])
for true, pred in zip(all_labels, all_preds):
    pred_table.add_data(emotion_labels[true], emotion_labels[pred], int(pred))
run.log({"predictions": pred_table})

run.finish()

best_val_acc,58.95158
epoch,20
final_best_epoch,17
final_best_val_acc,58.95158
lr,1e-05
train_acc,86.08438
train_loss,0.41419
val_acc,58.63811
val_loss,1.45326


In [23]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

class FERTestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.pixels = dataframe['pixels'].values
        self.transform = transform

    def __len__(self):
        return len(self.pixels)

    def __getitem__(self, idx):
        pixels = np.array(self.pixels[idx].split(), dtype=np.float32)
        image = pixels.reshape(48, 48) / 255.0
        image = torch.tensor(image).unsqueeze(0)

        if self.transform:
            image = self.transform(image)

        return image

test_loader = DataLoader(
    FERTestDataset(test_df),
    batch_size=32,
    shuffle=False
)

In [24]:
test_preds = []
test_probs = []

deep_model.eval()

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        outputs = deep_model(images)
        probs = torch.softmax(outputs, dim=1)

        _, predicted = outputs.max(1)

        test_preds.extend(predicted.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())

test_preds = np.array(test_preds)
test_probs = np.array(test_probs)

print(f"Generated {len(test_preds)} predictions")

Generated 7178 predictions


In [25]:
import pandas as pd

submission = pd.DataFrame(test_preds)
submission_path = "/content/drive/MyDrive/ml_assn_04/submission.csv"

submission.to_csv(
    submission_path,
    index=False,
    header=False
)

print(submission.head())

   0
0  4
1  4
2  0
3  4
4  3


In [26]:
import wandb

run = wandb.init(
    entity="ml_assn_04",
    project="ml_assn_04",
    id="001zqj9i",
    resume="must"
)

run.log({
    "test_prediction_distribution": wandb.Histogram(test_preds)
})

table = wandb.Table(
    columns=[
        "row_idx",
        "predicted_label",
        "predicted_class",
        "confidence"
    ]
)

for i, (pred, prob) in enumerate(zip(test_preds, test_probs)):
    table.add_data(
        i,
        emotion_labels[pred],
        int(pred),
        float(prob[pred])
    )

run.log({
    "test_predictions": table
})

artifact = wandb.Artifact(
    name="FER_submission",
    type="submission"
)

artifact.add_file(submission_path)
run.log_artifact(artifact)
run.finish()

best_val_acc,58.95158
epoch,20
final_best_epoch,17
final_best_val_acc,58.95158
lr,1e-05
train_acc,86.08438
train_loss,0.41419
val_acc,58.63811
val_loss,1.45326


In [28]:
run = wandb.init(
    entity="ml_assn_04",
    project="ml_assn_04",
    id="001zqj9i",
    resume="must"
)

model_artifact = wandb.Artifact(
    name="best_deep-cnn-cosine-lr",
    type="model"
)

model_artifact.add_file(
    "/content/drive/MyDrive/ml_assn_04/best_deep-cnn-cosine-lr.pt"
)

run.log_artifact(model_artifact)
run.finish()

best_val_acc,58.95158
epoch,20
final_best_epoch,17
final_best_val_acc,58.95158
lr,1e-05
train_acc,86.08438
train_loss,0.41419
val_acc,58.63811
val_loss,1.45326


wandb: WARNING Artifact "best_deep-cnn-cosine-lr" already exists with the same content. No new version will be created.


best_val_acc,58.95158
epoch,20
final_best_epoch,17
final_best_val_acc,58.95158
lr,1e-05
train_acc,86.08438
train_loss,0.41419
val_acc,58.63811
val_loss,1.45326
